# Chapter 10: Deploying Agents

This notebook mirrors the chapter. Run the cells in order, from the `Chapter10`
folder. Nothing here needs an API key. The SDK runner, the tools, the approval
pause, the state store, the pipeline, and the register are real. Only the model's
replies are scripted, which also makes every result below the same on every run.

All the agents, figures, and records are illustrative.

Pinned version: openai-agents 0.24.0.

In [1]:
import sys
from importlib.metadata import version
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from examples import (
    board,
    bundle_show,
    buy_route,
    contract_demo,
    intake_demo,
    listing_demo,
    pause_resume,
    pipeline_run,
    rollout_demo,
    switch_drill,
)

print("openai-agents", version("openai-agents"))

openai-agents 0.24.0


## Choosing where an agent runs

A run stops at an approval and is parked in a table the agent cannot write to.
A second worker, with no memory of the first, resumes it after the planner
approves. The same approval cannot be used twice, and an approval given under
an old release is void.

In [2]:
lines = await pause_resume.main(scripted_run=True)

worker A, release maintenance-planner@41.3
  status       waiting
  asks for     create_work_order: P-207, Replace pump bearings, BK-6207
  work orders  0
worker B, a new process, after the planner approves
  status       done
  output       The work order is created. P-207 is past its alarm limit.
  work orders  1 (WO-4100)
  same approval sent twice: gone, unknown run, or already decided
a second run waits while the release moves to 41.4
  status       stale, release moved while waiting
  work orders  1


## Packaging an agent: What goes in and stays out

A tool contract names what a server may be asked to do. The check runs before
the call, and the server makes its own decision afterward. The second half of
the output is what happens when somebody loosens the contract by mistake.

In [3]:
lines = await contract_demo.main()

contract parts-catalog@3.2.0
  allow  search_parts, get_lead_time
  never  create_order, update_price
the agent searches the catalog, then tries to place an order
  contract       parts-catalog@3.2.0 search_parts
  policy_denied  parts-catalog.create_order: outside contract 3.2.0
  the agent's reply: I found the part. I am not permitted to order it.
the same request after someone loosens the contract by mistake
  contract       parts-catalog@3.2.0 search_parts
  contract       parts-catalog@3.2.0 create_order
  server_refused parts-catalog: no write scope for maintenance-planner


## Building a reproducible release bundle and evaluation gate

The bundle is one file that names everything that changes how the agent
behaves. The pipeline checks that every pointer in it still matches the file it
names, and one digest names the release.

In [4]:
lines = bundle_show.main()

bundle: maintenance-planner@41.3
image: registry.internal/maintenance-planner@sha256:9f3a0f42b37d
sandbox: registry.internal/planner-sandbox@sha256:8de5358e155c
prompts: prompts/maintenance.txt@feca11568953
contracts: {parts-catalog: 3.2.0, work-orders: 1.4.0}
policy_pack: policy@1.9.0
model:
  primary: {id: gpt-6.1-terra, effort: low}
  fallback: {id: gpt-6-luna, effort: low, golden: evals/fallback.json}
memory_schema: {version: 4, reads: [3, 4]}
eval_suite: golden@41

release digest  sha256:8bd6abe5e53f
pointers        all match the files they name


The pipeline runs two attempts at the same release. The first attempt regressed,
and only block mode stops it. A committed key stops the pipeline in every mode,
because mandatory checks are never part of the gate's rollout.

In [5]:
lines = pipeline_run.main()

first attempt at 41.3: a prompt change
  monitor  exit 0  block: quality fell 13.3%, margin 12.4% (recorded only)
  comment  exit 0  block: quality fell 13.3%, margin 12.4% (posted as a comment)
  block    exit 1  block: quality fell 13.3%, margin 12.4% (enforced)
a key committed by mistake, with the gate still in monitor mode
  monitor  exit 1  floor: config.env: provider key
second attempt at 41.3, in block mode
bundle maintenance-planner@41.3  digest 8bd6abe5e53f
  floor      ok    no secrets
  bundle     ok    every pointer matches its file
  contracts  ok    agent identity, egress declared
  gate       ok    pass: no fall against the baseline (enforced)
  exit code 0
  record     verdict pass, report sha256 7118d82bc8b4


A canary moves one plant to the new bundle, and a rollback moves it back. The
last three lines are what the rollback did not undo.

In [6]:
lines = await rollout_demo.main()

before
  plant-1 41.2  plant-2 41.2  plant-3 41.2  plant-4 41.2
canary: plant-2 runs maintenance-planner@41.3
  plant-1 41.2  plant-2 41.3  plant-3 41.2  plant-4 41.2
rollback: plant-2 returns to maintenance-planner@41.2
  plant-1 41.2  plant-2 41.2  plant-3 41.2  plant-4 41.2
what the rollback left behind
  memory: 1 written in schema version 4
  side effects: work orders WO-4100 still exist
  runs in flight: 1 waiting for approval


## Stopping agents safely in production

Six switches, each pulled on its own copy of staging, and then a probe that tries
the thing that should have stopped.

In [7]:
lines = switch_drill.main()

switch         owner              what it stops             drill
gateway        platform on-call   model calls               stopped
identity       security on-call   every tool and data call  stopped
tool contract  tool owner         one operation             stopped
runtime        platform on-call   new runs                  stopped
register       agent owner        new adoption              stopped
data supply    data owner         one data product          stopped


## Scaling the platform with one path, register, and dashboard

The register is compared with the systems an agent cannot avoid, and then the
maintenance planner earns its listing on the third attempt.

In [8]:
lines = listing_demo.main()

register holds 1 agent: maintenance-planner
  gateway key 'docs-assistant-prod' has no register entry
  provider invoice 'direct API account, sales engineering' has no register entry
  vendor feature 'purchasing suite: AI suggestions' has no register entry
listing the maintenance planner, tier high
  attempt 1  refused: missing evidence: oversight_note
  attempt 2  refused: eval_report was produced under a different bundle
  attempt 3  listed


Three proposals go through intake. Only one becomes a build.

In [9]:
lines = intake_demo.main()

shift-handover  reuse  maintenance-planner already covers 100% of this
expense-coding  buy    finance suite assistant is licensed and covers 100%
parts-quote     build  nothing existing fits
                       scaffold: agent-parts-quote, tier medium
                       gate in monitor mode, golden suite of 0 tasks


## Choosing an adoption route: Build, join, assemble, or buy

A bought agent's gap sheet, read at its own tier and then at the highest tier
with a contract term where a control should be.

In [10]:
lines = buy_route.main()

purchasing-advisor, tier medium
  identification    met
  security_review   met
  action_limits     met
  audit_records     compensating_control
  lifecycle_notice  contract
  usage_evidence    risk_acceptance
  blocking gaps: none
the same product, if it placed orders itself and offered only promises
  audit_records: a contract line is not a control at this tier
  usage_evidence: nobody has accepted this risk by name


## Real-world pattern: Reviewing four agents on one dashboard

One row for each agent, read from its register entry and the records the entry
points at. The joined service's result comes through the adapter.

In [11]:
lines = board.main()

agent                route    stage           quality           value      drift
maintenance-planner  build    steady 7/7      97% of 30 tasks   measured   5 of 5
document-assistant   join     listed 5/6      90% of 40 tasks   baselined  4 of 5
parts-quote          assemble live 5/7        92% of 13 tasks   measured   4 of 5
purchasing-advisor   buy      contracted 3/6  100% of 12 tasks  declared   1 of 5

agent                cost last month     value claim
maintenance-planner  $2,120 metered      work orders approved 1.5 days earlier
document-assistant   $1,135 metered      procedure found in 2 minutes, not 15
parts-quote          $53 metered         quotes sent the same day
purchasing-advisor   $1,050 allocated    buyer review time down 20 percent

drift signals: 14 of 20 reporting, 4 not configured, 2 not observable
joined suite docqa@112: 40 cases, 3 attempts each, read through the adapter


## Running against a live model

Set `OPENAI_API_KEY` and run the pause and resume example without `--scripted`:

    python -m examples.pause_resume

The planner then uses the model pinned in `template-repository/bundle.yaml`.